# MCP ❤️ Microsoft Web IQ through API Management

Connect an MCP client to the [Microsoft Web IQ MCP server](https://webiq.microsoft.ai/documentation/mcp/) through Azure API Management (APIM), discover the tools enabled for your Web IQ account, invoke the `web` tool, and attribute usage to an APIM subscription. The gateway also prevents MCP clients from bypassing the lab's Browse restriction.

**Audience:** Developers and platform teams governing web-grounding tools for agents.

**Prerequisites:**

- Complete the deployment steps in [web-iq.ipynb](web-iq.ipynb) after the `/mcp` operations have been added.
- Install the repository environment with `uv sync`, then select that Python kernel.
- Sign in with Azure CLI and retain access to the lab resource group.
- A Web IQ API key from Profile Management; optionally, configure Entra ID for the alternate authentication step.

**Learning goals:**

- Use the MCP streamable HTTP transport through an APIM endpoint.
- Discover account-scoped Web IQ tools and invoke `web`.
- Use a caller-provided Web IQ API key or optional Entra ID pass-through.
- Enforce a `browse` tool denial and query per-tool usage metrics.

```mermaid
flowchart LR
    Client[MCP notebook client]
    MicrosoftEntra[Microsoft Entra ID]
    subgraph Gateway[Azure API Management]
        Subscription[Validate APIM subscription]
        RequestMetric[Emit request metric<br/>MCP Tool dimension]
        Inspect{JSON-RPC request?}
        Browse{tools/call browse?}
        BlockMetric[Emit blocked-request metric]
        Forbidden[Return structured 403]
        Auth{Web IQ credential present?}
        Unauthorized[Return structured 401]
        Entra[Pass through bearer token]
        ResponseMetric[Emit response and latency metrics]
    end
    subgraph WebIQ[Microsoft Web IQ MCP server]
        Protocol[initialize and tools/list]
        Web[web tool]
        Other[Other account-enabled tools]
    end
    Client -.->|Client credentials<br/>Web IQ scope| MicrosoftEntra
    MicrosoftEntra -.->|Optional bearer token| Client
    Client -->|POST GET DELETE /mcp<br/>APIM key plus x-apikey or bearer| Subscription
    Subscription --> RequestMetric --> Inspect --> Browse
    Browse -->|Yes| BlockMetric --> Forbidden --> Client
    Browse -->|No| Auth
    Auth -->|Caller x-apikey| Protocol
    Auth -->|Bearer| Entra --> Protocol
    Auth -->|Missing| Unauthorized --> Client
    Protocol --> Web
    Protocol --> Other
    Web --> ResponseMetric --> Client
    Other --> ResponseMetric
    RequestMetric -.-> Insights[Application Insights]
    BlockMetric -.-> Insights
    ResponseMetric -.-> Insights
    Insights --> Logs[Log Analytics]
```


## Outline

1. Retrieve the existing Web IQ lab deployment.
2. Connect over streamable HTTP and discover tools.
3. Invoke the Web Search MCP tool.
4. Optionally switch upstream authentication to Entra ID.
5. Verify that the Browse policy also applies to MCP.
6. Query per-tool APIM metrics and try an account-enabled tool.


<a id='deployment'></a>
### 1️⃣ Retrieve the deployed gateway

This notebook intentionally reuses the resources created by `web-iq.ipynb`. Redeploy that notebook's Bicep step after changing `openapi.json` or `policy.xml`; otherwise the deployed APIM API will not contain `/mcp`. Deployment outputs provide the gateway URL and APIM subscription keys; the Web IQ credential is supplied separately by the caller and is not stored in APIM.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from getpass import getpass

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

sys.path.insert(1, '../../shared')
import utils

deployment_name = 'web-iq'
resource_group_name = f'lab-{deployment_name}'

output = utils.run(
    f'az deployment group show --name {deployment_name} --resource-group {resource_group_name}',
    f"Retrieved deployment '{deployment_name}'",
    f"Failed to retrieve deployment '{deployment_name}'",
)
if not output.success or not output.json_data:
    raise RuntimeError('Deploy the lab with web-iq.ipynb before running this notebook.')

apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM gateway URL')
application_insights_name = utils.get_deployment_output(output, 'applicationInsightsName', 'Application Insights name')
web_iq_api_path = utils.get_deployment_output(output, 'webIqApiPath', 'Web IQ API path')
apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("'", '"'))

web_iq_api_key = os.getenv('WEBIQ_API_KEY') or getpass('Microsoft Web IQ API key: ')
if not web_iq_api_key:
    raise ValueError('Set WEBIQ_API_KEY or enter a Web IQ API key when prompted.')

mcp_endpoint = f'{apim_resource_gateway_url}/{web_iq_api_path}/mcp'
apim_headers = {
    'Ocp-Apim-Subscription-Key': apim_subscriptions[0]['key'],
    'x-apikey': web_iq_api_key,
}

utils.print_ok(f'MCP endpoint: {mcp_endpoint}')
utils.print_info(f"Consumer: {apim_subscriptions[0]['displayName']} (****{apim_subscriptions[0]['key'][-4:]})")


<a id='discover'></a>
### 2️⃣ Discover Web IQ MCP tools through APIM

The MCP client initializes a streamable HTTP session and sends `tools/list` through APIM. The APIM subscription identifies the consumer, while the caller-provided `x-apikey` authenticates to Web IQ. APIM strips its own subscription credential and forwards the Web IQ credential unchanged. Web IQ returns only the tools permitted for that account.

The upstream capability list can still advertise `browse`. Enforcement happens on invocation: the APIM policy inspects `tools/call` and rejects `browse` before forwarding it.


In [ ]:
async def list_web_iq_tools(headers: dict[str, str]):
    async with streamablehttp_client(mcp_endpoint, headers=headers, timeout=60.0) as streams:
        async with ClientSession(streams[0], streams[1]) as session:
            await session.initialize()
            response = await session.list_tools()
            return response.tools

tools = await list_web_iq_tools(apim_headers)
available_tool_names = {tool.name for tool in tools}

utils.print_ok(f'Discovered {len(tools)} Web IQ tool(s) through APIM')
for tool in tools:
    print(f'  - {tool.name}: {tool.description or "No description"}')

if 'web' not in available_tool_names:
    raise RuntimeError("The Web IQ account does not expose the required 'web' tool.")


<a id='web-tool'></a>
### 3️⃣ Invoke the `web` tool

MCP uses the same Web Search parameters as the REST API. The result is returned as MCP content, so the helper below extracts text safely and truncates only the notebook display—not the response sent by Web IQ. This call produces gateway metrics with `Operation ID = mcp-post` and `MCP Tool = web`.


In [ ]:
async def call_web_iq_tool(tool_name: str, arguments: dict, headers: dict[str, str]):
    async with streamablehttp_client(mcp_endpoint, headers=headers, timeout=60.0) as streams:
        async with ClientSession(streams[0], streams[1]) as session:
            await session.initialize()
            return await session.call_tool(tool_name, arguments)

def render_mcp_content(result, limit: int = 4000) -> str:
    chunks = []
    for item in result.content:
        text = getattr(item, 'text', None)
        chunks.append(text if text is not None else str(item))
    rendered = '\n'.join(chunks)
    return rendered if len(rendered) <= limit else rendered[:limit] + '\n… [display truncated]'

web_result = await call_web_iq_tool(
    'web',
    {
        'query': 'What is Azure API Management and how does it support AI gateways?',
        'maxResults': 3,
        'maxLength': 3000,
        'contentFormat': 'markdown',
    },
    apim_headers,
)

if getattr(web_result, 'isError', False):
    raise RuntimeError(render_mcp_content(web_result))
print(render_mcp_content(web_result))


<a id='entra-id'></a>
### 4️⃣ Optional: use Entra ID for Web IQ authentication

Set `WEBIQ_TENANT_ID`, `WEBIQ_CLIENT_ID`, and `WEBIQ_CLIENT_SECRET` for an app registration bound in Web IQ Profile Management. APIM still requires its subscription key to identify the client, but it detects the Web IQ bearer token, removes `x-apikey`, and passes the token upstream. The requested scope is `https://api.microsoft.ai/.default`.

If these variables are absent, the cell skips cleanly and the API-key-backed connection remains the demonstrated path.


In [ ]:
from msal import ConfidentialClientApplication

entra_settings = {
    'tenant_id': os.getenv('WEBIQ_TENANT_ID'),
    'client_id': os.getenv('WEBIQ_CLIENT_ID'),
    'client_secret': os.getenv('WEBIQ_CLIENT_SECRET'),
}

if not all(entra_settings.values()):
    utils.print_info(
        'Optional Entra ID MCP connection skipped. Set WEBIQ_TENANT_ID, WEBIQ_CLIENT_ID, and WEBIQ_CLIENT_SECRET to run it.'
    )
else:
    entra_client = ConfidentialClientApplication(
        client_id=entra_settings['client_id'],
        client_credential=entra_settings['client_secret'],
        authority=f"https://login.microsoftonline.com/{entra_settings['tenant_id']}",
    )
    token_result = entra_client.acquire_token_for_client(
        scopes=['https://api.microsoft.ai/.default']
    )
    if 'access_token' not in token_result:
        raise RuntimeError(token_result.get('error_description', 'Could not acquire a Web IQ access token.'))

    entra_headers = {
        'Ocp-Apim-Subscription-Key': apim_subscriptions[0]['key'],
        'Authorization': f"Bearer {token_result['access_token']}",
    }
    entra_tools = await list_web_iq_tools(entra_headers)
    utils.print_ok(f'Entra ID connection discovered {len(entra_tools)} tool(s) through APIM')


<a id='browse-policy'></a>
### 5️⃣ Verify that MCP cannot bypass the Browse policy

The policy parses only MCP `tools/call` messages and compares the bounded tool name. A call to `browse` should fail with HTTP `403` before Web IQ receives it, even when `tools/list` advertised that capability. Search queries, URLs, response content, credentials, and session IDs are not emitted as metric dimensions.


In [ ]:
def exception_messages(error: BaseException) -> list[str]:
    nested = getattr(error, 'exceptions', None)
    if nested:
        messages = []
        for child in nested:
            messages.extend(exception_messages(child))
        return messages
    return [f'{type(error).__name__}: {error}']

browse_was_blocked = False
try:
    await call_web_iq_tool(
        'browse',
        {
            'url': 'https://news.microsoft.com/source/',
            'contentFormat': 'markdown',
            'maxLength': 3000,
        },
        apim_headers,
    )
except Exception as error:
    browse_was_blocked = True
    utils.print_ok('Browse was blocked by APIM as expected')
    print('\n'.join(exception_messages(error))[:1200])

assert browse_was_blocked, 'Browse unexpectedly passed through APIM. Verify that the latest policy is deployed.'


<a id='metrics'></a>
### 6️⃣ Query MCP tool usage

Custom metrics can take several minutes to appear. MCP initialization and capability discovery are recorded as `protocol`; `tools/call` messages use the requested tool name. The query separates allowed and blocked messages by APIM subscription, authentication mode, and MCP tool.


In [ ]:
import pandas as pd

mcp_usage_query = r'''
customMetrics
| where timestamp > ago(1h) and name in ('Web IQ Requests', 'Web IQ Blocked Requests')
| extend dimensions = todynamic(customDimensions)
| extend SubscriptionId = tostring(dimensions['Subscription ID'])
| extend OperationId = tostring(dimensions['Operation ID'])
| extend Authentication = tostring(dimensions['Authentication'])
| extend McpTool = tostring(dimensions['MCP Tool'])
| where OperationId startswith 'mcp-'
| summarize Requests = sumif(value, name == 'Web IQ Requests'),
    BlockedRequests = sumif(value, name == 'Web IQ Blocked Requests')
    by SubscriptionId, OperationId, Authentication, McpTool
| order by Requests desc
'''

result = utils.run(
    f'az monitor app-insights query --app {application_insights_name} '
    f'--resource-group {resource_group_name} --analytics-query {json.dumps(mcp_usage_query)}',
    'Application Insights query succeeded',
    'Application Insights query failed',
)

if result.success and result.json_data.get('tables'):
    table = result.json_data['tables'][0]
    mcp_usage_df = pd.DataFrame(
        table.get('rows', []),
        columns=[column['name'] for column in table.get('columns', [])],
    )
else:
    mcp_usage_df = pd.DataFrame()

mcp_usage_df


### Exercise — invoke another account-enabled tool

Predict whether `news` is available from the discovered list, then run the scaffold. Change the query or choose another non-Browse tool from `available_tool_names`. Web IQ scopes the tool list to the account, so a missing tool is expected rather than a gateway error.


In [ ]:
exercise_tool = 'news'

if exercise_tool not in available_tool_names:
    utils.print_info(f"'{exercise_tool}' is not enabled for this Web IQ account. Choose from: {sorted(available_tool_names)}")
else:
    # Exercise: change the query and compare the MCP Tool metric after telemetry arrives.
    exercise_result = await call_web_iq_tool(
        exercise_tool,
        {
            'query': 'Latest Azure API Management announcements',
            'maxResults': 3,
        },
        apim_headers,
    )
    print(render_mcp_content(exercise_result))


### Pitfalls and extensions

- **Redeployment:** APIM learns `/mcp` and the MCP-aware policy only after rerunning the Bicep deployment.
- **Account scope:** Web IQ exposes only tools allowed for the credential used on that session.
- **Transport methods:** Streamable HTTP can use `POST`, `GET`, and `DELETE`; all three are published in APIM.
- **Browse discovery versus authorization:** `tools/list` is upstream-owned and may advertise `browse`; APIM remains the enforcement point on `tools/call`.
- **Telemetry:** A single logical task produces protocol messages as well as tool calls. Use the `MCP Tool` dimension to distinguish them.
- **Credential boundaries:** The client sends an APIM subscription key plus either a Web IQ `x-apikey` or bearer token. APIM strips its subscription credential and never stores the Web IQ API key.
- **Production hardening:** Add per-subscription rate limits and consider validating client JWTs at APIM in addition to upstream Web IQ authentication.


<a id='clean-up'></a>
### 🗑️ Clean up resources

When you finish both notebooks, run [clean-up-resources.ipynb](clean-up-resources.ipynb) to remove the lab resource group and avoid further charges.
